# Extract embeddings from ESM Cambrian for fast-folding proteins

This notebook shows how to extract sequence embeddings using [ESM Cambrian (ESMC)](https://github.com/evolutionaryscale/esm), which are
required as input to **PLaTITO** .

Both models are accessed via the [EvolutionaryScale Forge API](https://forge.evolutionaryscale.ai), which requires an API token.

| Model | Forge model name | Embedding dim |
|-------|-----------------|---------------|
| **ESMC 300M** | `esmc-300m-2024-12` | 960 |
| **ESMC 6B**  | `esmc-6b-2024-12` | 2560 |

The 300M model is needed for **PLaTITO 3M** and the 6B model is needed for **PLaTITO 19M** and **PLaTITO 36M**.

---

Embeddings are saved as `.pt` files of shape `[L, D]`, where `L` is the number of residues and `D` is the embedding dimension.

In [1]:
fast_folders_sequences = {
    "bba":         "EQYTAKYKGRTFRNEKELRDFIEKFKGR",
    "villin":      "LSDEDFKAVFGMTRSAFANLPLWLQQHLLKEKGLF",
    "trp_cage":    "DAYAQWLADGGPSSGRPPPS",
    "bbl":         "GSQNNDALSPAIRRLLAEWNLDASAIKGTGVGGRLTREDVEKHLAKA",
    "a3d":         "MGSWAEFKQRLAAIKTRLQALGGSEAELAAFEKEIAAFESELQAYKGKGNPEVEALRKEAAAIRDELQAYRHN",
    "wwdomain":    "GSKLPPGWEKRMSRDGRVYYFNHITGTTQFERPSG",
    "ntl9":        "MKVIFLKDVKGMGKKGEIKNVADGYANNFLFKQGLAIEA",
    "protein_g":   "DTYKLVIVLNGTTFTYTTEAVDAATAEKVFKQYANDAGVDGEWTYDAATKTFTVTE",
    "protein_b":   "LKNAIEDAIAELKKAGITSDFYFNAINKAKTVEEVNALVNEILKAHA",
    "homeodomain": "MKQWSENVEEKLKEFVKRHQRITQEELHQYAQRLGLNEEAIRQFFEEFEQRK",
    "lambda":      "PLTQEQLEAARRLKAIWEKKKNELGLSYESVADKMGMGQSAVAALFNGINALNAYNAALLAKILKVSVEEFSPSIAREIY",
}

In [3]:
FORGE_TOKEN = "<your forge token>"

## ESMC 300M

In [4]:
import os
import torch
from esm.sdk.forge import ESM3ForgeInferenceClient
from esm.sdk.api import ESMProtein, LogitsConfig

forge_client = ESM3ForgeInferenceClient(
    model="esmc-300m-2024-12",
    url="https://forge.evolutionaryscale.ai",
    token=FORGE_TOKEN,
)

In [5]:
out_dir = "embeddings/esmc_300m"
os.makedirs(out_dir, exist_ok=True)

embeddings_dict = {}

for name, sequence in fast_folders_sequences.items():
    protein = ESMProtein(sequence=sequence)
    protein_tensor = forge_client.encode(protein)
    logits_output = forge_client.logits(
        protein_tensor, LogitsConfig(sequence=True, return_embeddings=True, return_hidden_states=True)
    )
    # Use layer 20; strip BOS/EOS tokens → shape [L, D]
    emb = logits_output.hidden_states.squeeze(1)[20][1:-1, :].cpu()
    assert emb.shape[0] == len(sequence), (
        f"{name}: embedding length {emb.shape[0]} != sequence length {len(sequence)}"
    )
    torch.save(emb, os.path.join(out_dir, f"{name}.pt"))
    print(f"  {name:20s}  shape={tuple(emb.shape)}")

/dtu/p1/panant/miniconda3/envs/platito/lib/python3.11/site-packages/esm/utils/misc.py:275: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x)


  bba                   shape=(28, 960)
  villin                shape=(35, 960)
  trp_cage              shape=(20, 960)
  bbl                   shape=(47, 960)
  a3d                   shape=(73, 960)
  wwdomain              shape=(35, 960)
  ntl9                  shape=(39, 960)
  protein_g             shape=(56, 960)
  protein_b             shape=(47, 960)
  homeodomain           shape=(52, 960)
  lambda                shape=(80, 960)


## ESMC 6B

In [6]:
forge_client = ESM3ForgeInferenceClient(
    model="esmc-6b-2024-12",
    url="https://forge.evolutionaryscale.ai",
    token=FORGE_TOKEN,
)

In [7]:
out_dir = "embeddings/esmc_6b"
os.makedirs(out_dir, exist_ok=True)

embeddings_dict = {}

for name, sequence in fast_folders_sequences.items():
    protein = ESMProtein(sequence=sequence)
    protein_tensor = forge_client.encode(protein)
    logits_output = forge_client.logits(
        protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
    )
    # Strip BOS/EOS tokens → shape [L, D]
    emb = logits_output.embeddings[0, 1:-1, :].cpu()
    assert emb.shape[0] == len(sequence), (
        f"{name}: embedding length {emb.shape[0]} != sequence length {len(sequence)}"
    )
    torch.save(emb, os.path.join(out_dir, f"{name}.pt"))
    print(f"  {name:20s}  shape={tuple(emb.shape)}")

  bba                   shape=(28, 2560)
  villin                shape=(35, 2560)
  trp_cage              shape=(20, 2560)
  bbl                   shape=(47, 2560)
  a3d                   shape=(73, 2560)
  wwdomain              shape=(35, 2560)
  ntl9                  shape=(39, 2560)
  protein_g             shape=(56, 2560)
  protein_b             shape=(47, 2560)
  homeodomain           shape=(52, 2560)
  lambda                shape=(80, 2560)


---

## Output layout

After running either option the output directory will contain:

```
embeddings/
└── esmc_300m/           # or esmc_6b/
    ├── bba.pt           
    ├── villin.pt
    ├── ...
    └── a3d.pt
```